# 01 — Побудова optical-flow velocity з raw MCAP

Цей notebook відтворює preprocessing із експериментальної Colab-чернетки у керованій локальній формі:

1. thermal frames → sparse Lucas–Kanade flow;
2. фізичні та MAD-фільтри pixel flow;
3. IMU gyro compensation + camera calibration + rangefinder → швидкість у FRD;
4. лише для повного запуску — один новий MCAP із вихідними повідомленнями та двома vision topics.

Використовуйте kernel **Project CV (ROS 2 Humble)** і запускайте клітинки зверху вниз. Notebook нічого не встановлює та не змінює fusion-алгоритм.

## Контракт даних

- `PROJECT_CV_RAW_BAG` — незмінний сирий вхід; сюди нічого не записується.
- `PROJECT_CV_DERIVED_BAG` — незмінний golden/reference MCAP викладача; він потрібен лише для порівняння.
- усі нові CSV, NPZ, JSON та MCAP створюються тільки під `PROJECT_CV_ARTIFACTS`.

Golden-файл є орієнтиром, а не доказом правильності алгоритму. Збіг кількості повідомлень не доводить збіг timestamps або числових значень.

In [ ]:
import json
import os
import platform
import sys
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import JSON, display

assert platform.system() == "Linux", (
    "Select the 'Project CV (ROS 2 Humble)' kernel; preprocessing must run in WSL Ubuntu."
)
assert os.environ.get("ROS_DISTRO") == "humble", os.environ.get("ROS_DISTRO")

required_environment = (
    "PROJECT_CV_SOURCE",
    "PROJECT_CV_RAW_BAG",
    "PROJECT_CV_DERIVED_BAG",
    "PROJECT_CV_CALIBRATION",
    "PROJECT_CV_ARTIFACTS",
)
missing_environment = [name for name in required_environment if not os.environ.get(name)]
assert not missing_environment, f"Missing environment variables: {missing_environment}"

PROJECT_CV_SOURCE = Path(os.environ["PROJECT_CV_SOURCE"]).resolve()
PROJECT_CV_SRC = PROJECT_CV_SOURCE / "src"
assert PROJECT_CV_SRC.is_dir(), f"Source package directory is missing: {PROJECT_CV_SRC}"
if str(PROJECT_CV_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_CV_SRC))

from project_cv.preprocessing_validation import validate_preprocessing_bags
from project_cv.vision_velocity import (
    LEGACY_BASELINE_NOTES,
    PreprocessingArtifacts,
    VisionVelocityConfig,
    build_derived_bag,
    compute_gyrocompensated_velocity,
    compute_sparse_lk,
    filter_sparse_lk,
)

print("Python:", platform.python_version())
print("ROS_DISTRO:", os.environ["ROS_DISTRO"])
print("Project source package:", PROJECT_CV_SRC)

In [ ]:
RAW_BAG_DIR = Path(os.environ["PROJECT_CV_RAW_BAG"]).resolve()
GOLDEN_BAG_DIR = Path(os.environ["PROJECT_CV_DERIVED_BAG"]).resolve()
CALIBRATION_DIR = Path(os.environ["PROJECT_CV_CALIBRATION"]).resolve()
ARTIFACTS_ROOT = Path(os.environ["PROJECT_CV_ARTIFACTS"]).resolve()

required_inputs = (
    RAW_BAG_DIR / "metadata.yaml",
    GOLDEN_BAG_DIR / "metadata.yaml",
    CALIBRATION_DIR / "thermal_camera.yml",
    CALIBRATION_DIR / "cam_to_imu_rot_mtrx.yml",
)
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
assert not missing_inputs, f"Missing runtime inputs: {missing_inputs}"
assert ARTIFACTS_ROOT.is_dir(), ARTIFACTS_ROOT

display(
    pd.DataFrame(
        [
            {"role": "immutable raw input", "path": str(RAW_BAG_DIR)},
            {"role": "immutable golden reference", "path": str(GOLDEN_BAG_DIR)},
            {"role": "read-only calibration", "path": str(CALIBRATION_DIR)},
            {"role": "generated artifacts only", "path": str(ARTIFACTS_ROOT)},
        ]
    )
)

## Важливо: спочатку відтворюємо legacy baseline

Цей етап навмисно не «виправляє» сумнівні припущення старої чернетки. Спочатку потрібен відтворюваний baseline і чесне порівняння з golden MCAP; зміни FOV, timing, height model, undistortion і фільтрів належать до наступного експериментального етапу.

In [ ]:
for index, note in enumerate(LEGACY_BASELINE_NOTES, start=1):
    print(f"{index}. {note}")

GOLDEN_COUNT_ANCHORS = {
    "copied_source_messages": 654_089,
    "flow_messages_written": 76_966,
    "velocity_messages_written": 59_859,
    "total_messages": 790_914,
}
print("\nGolden count anchors come from the teacher-provided metadata; they are count-only checks.")

## Режим запуску

- `smoke` — безпечний швидкий тест короткого вікна; створює проміжні CSV/JSON, але **не** будує MCAP.
- `full` — рахує весь політ з нуля та наприкінці будує повний derived MCAP. Перший запуск може бути тривалим і потребує кількох GB вільного місця.
- `reuse` — повний режим, але повторно використовує лише артефакти, JSON fingerprint яких точно відповідає поточній конфігурації. Відсутні етапи будуть дораховані.

`smoke` є безпечним default для `Run All`. Для реальної повної побудови змініть `MODE` на `"full"` або задайте `PROJECT_CV_PREPROCESS_MODE=full`, перезапустіть kernel і виконайте `Run All`. Немає другого прихованого прапорця підтвердження.

In [ ]:
MODE = os.environ.get("PROJECT_CV_PREPROCESS_MODE", "smoke").strip().lower()  # "smoke", "full" або "reuse"
SMOKE_START_OFFSET_SEC = 145.0  # ділянка початку руху з валідним rangefinder
SMOKE_DURATION_SEC = 15.0
OVERWRITE_DERIVED_BAG = False  # захист від випадкового видалення вже створеного MCAP

assert MODE in {"smoke", "full", "reuse"}, MODE
if MODE == "smoke":
    artifact_name = "preprocess_smoke"
    start_offset_sec = SMOKE_START_OFFSET_SEC
    duration_sec = SMOKE_DURATION_SEC
    reuse_existing = False
    build_final_bag = False
else:
    artifact_name = "preprocess_v1"
    start_offset_sec = 0.0
    duration_sec = None
    reuse_existing = MODE == "reuse"
    build_final_bag = True

config = VisionVelocityConfig.from_environment(
    artifact_name=artifact_name,
    start_offset_sec=start_offset_sec,
    duration_sec=duration_sec,
)
artifacts = PreprocessingArtifacts.under(config.artifact_dir)

for protected_dir in (RAW_BAG_DIR, GOLDEN_BAG_DIR):
    assert protected_dir != config.artifact_dir.resolve()
    assert protected_dir not in config.artifact_dir.resolve().parents

print("MODE:", MODE)
print("Selected window:", "full bag" if duration_sec is None else f"{duration_sec:.1f} s")
print("Reuse fingerprint-matching outputs:", reuse_existing)
print("Build final MCAP:", build_final_bag)
print("Artifact directory:", artifacts.root)

In [ ]:
stage_timings_sec = {}

def run_timed_stage(name, operation):
    started = perf_counter()
    result = operation()
    elapsed = perf_counter() - started
    stage_timings_sec[name] = elapsed
    print(f"{name}: {elapsed:.2f} s")
    return result

def show_json_summary(title, path):
    path = Path(path)
    assert path.is_file(), f"Summary was not created: {path}"
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(f"{title}: {path}")
    display(JSON(data=payload, expanded=False))
    return payload

## Stage 1 — Sparse Lucas–Kanade

Декодує thermal frames, знаходить і трекає характерні точки, робить forward/backward check та записує raw/LPF pixel flow.

In [ ]:
lk_csv = run_timed_stage(
    "sparse_lk",
    lambda: compute_sparse_lk(config, reuse_existing=reuse_existing),
)
lk_summary = show_json_summary("Sparse LK summary", artifacts.lk_summary_json)

## Stage 2 — Legacy pixel-flow filter

Застосовує межі фізично допустимого руху, quality/dt gates, causal rolling MAD та IIR low-pass. Це baseline-фільтр із його відомими припущеннями.

In [ ]:
filtered_flow_csv = run_timed_stage(
    "physical_flow_filter",
    lambda: filter_sparse_lk(
        config,
        input_csv=lk_csv,
        reuse_existing=reuse_existing,
    ),
)
flow_summary = show_json_summary("Flow filter summary", artifacts.flow_summary_json)

## Stage 3 — Gyro compensation та FRD velocity

Перетворює pixel displacement через `fx/fy`, синхронізує IMU/range/Euler, компенсує обертання і переводить результат з осей камери у FRD.

In [ ]:
velocity_raw_csv, velocity_filtered_csv = run_timed_stage(
    "gyrocompensated_velocity",
    lambda: compute_gyrocompensated_velocity(
        config,
        filtered_flow_csv=filtered_flow_csv,
        reuse_existing=reuse_existing,
    ),
)
velocity_summary = show_json_summary("Velocity summary", artifacts.velocity_summary_json)

## Stage 4 — Derived MCAP і golden validation (лише `full`/`reuse`)

Повний режим одним проходом копіює raw messages і додає `/vision/lk_flow_px_filtered` та `/vision/velocity_frd`, після чого потоково порівнює обидва topics із golden MCAP. Smoke mode свідомо пропускає цей багатогігабайтний крок.

За замовчуванням існуючий generated MCAP не видаляється. Для нормального повторного запуску використовуйте `MODE = "reuse"`. `OVERWRITE_DERIVED_BAG = True` вмикайте лише якщо свідомо хочете перебудувати generated MCAP у каталозі artifacts. Raw і golden inputs цей прапорець не зачіпає.

In [ ]:
generated_bag_dir = None
bag_summary = None
golden_validation = None
if build_final_bag:
    generated_bag_dir = run_timed_stage(
        "derived_bag",
        lambda: build_derived_bag(
            config,
            filtered_flow_csv=filtered_flow_csv,
            filtered_velocity_csv=velocity_filtered_csv,
            reuse_existing=reuse_existing,
            overwrite=OVERWRITE_DERIVED_BAG,
        ),
    )
    bag_summary = show_json_summary("Derived bag summary", artifacts.bag_summary_json)
    golden_validation_path = artifacts.root / "golden_comparison.json"
    golden_validation = run_timed_stage(
        "golden_reference_validation",
        lambda: validate_preprocessing_bags(
            generated_bag_dir,
            GOLDEN_BAG_DIR,
            artifacts_root=ARTIFACTS_ROOT,
            diagnostics_path=golden_validation_path,
            numeric_tolerance=1e-7,
        ),
    )
    print("Golden comparison overall PASS:", golden_validation["overall"]["passed"])
    display(JSON(data=golden_validation, expanded=False))
else:
    print("Smoke mode: final MCAP intentionally skipped. Intermediate stages completed.")

## Підсумок та перевірка проти golden anchors

У full/reuse режимі Stage 4 уже виконує детальну потокову звірку schema, timestamps і числових полів. Нижче додатково показано компактне порівняння counts із metadata викладача. Статус `MATCH (count only)` у цій таблиці сам по собі не означає бітовий, часовий або числовий збіг; для цього дивіться `golden_comparison.json`.

In [ ]:
display(
    pd.DataFrame(
        [{"stage": name, "seconds": seconds} for name, seconds in stage_timings_sec.items()]
    )
)

if bag_summary is not None:
    generated_counts = {
        "copied_source_messages": int(bag_summary["copied_source_messages"]),
        "flow_messages_written": int(bag_summary["flow_messages_written"]),
        "velocity_messages_written": int(bag_summary["velocity_messages_written"]),
    }
    generated_counts["total_messages"] = sum(generated_counts.values())
    comparison_rows = []
    for metric, golden_value in GOLDEN_COUNT_ANCHORS.items():
        generated_value = generated_counts[metric]
        equal = generated_value == golden_value
        comparison_rows.append(
            {
                "metric": metric,
                "generated": generated_value,
                "golden_anchor": golden_value,
                "delta": generated_value - golden_value,
                "status": "MATCH (count only)" if equal else "DIFF — investigate",
            }
        )
    display(pd.DataFrame(comparison_rows))
    print("Count equality alone does NOT establish equality of timestamps or message values.")
else:
    print("Golden count comparison is skipped in smoke mode because no full MCAP was built.")

artifact_locations = {
    "artifact_root": artifacts.root,
    "sparse_lk_csv": artifacts.lk_raw_csv,
    "filtered_flow_csv": artifacts.flow_filtered_csv,
    "raw_velocity_csv": artifacts.velocity_raw_csv,
    "filtered_velocity_csv": artifacts.velocity_filtered_csv,
    "generated_bag": generated_bag_dir,
    "golden_comparison_json": None if golden_validation is None else artifacts.root / "golden_comparison.json",
}
display(
    pd.DataFrame(
        [
            {"artifact": name, "path": None if path is None else str(path)}
            for name, path in artifact_locations.items()
        ]
    )
)

print("Preprocessing run completed in mode:", MODE)

## Опційний повний запуск

Після успішного smoke test:

1. у клітинці **Режим запуску** встановіть `MODE = "full"` (або задайте змінну середовища `PROJECT_CV_PREPROCESS_MODE=full`);
2. перезапустіть kernel, щоб не змішувати стан двох конфігурацій;
3. виконайте `Run All`;
4. після завершення використовуйте `MODE = "reuse"` для безпечного повторного відкриття тих самих fingerprint-matching артефактів.

Повний generated bag буде у `$PROJECT_CV_ARTIFACTS/preprocess_v1/generated_with_velocity`. Golden bag викладача залишається незмінним. Далі запускайте `02_sparse_gps_fusion.ipynb`, а потім `03_fusion_diagnostics.ipynb`; модифікації fusion та висотного каналу виконуватимуться окремим етапом.